# 01 — DEA Fundamentals
## What DEA Measures and Why Isolated Facilities Break It

Companion notebook to the article [**The Method That Almost Works**](https://www.linkedin.com/pulse/method-almost-works-oghenero-inana-za0oe) — part of a series on operational efficiency, performance measurement, and analytical methods for industrial systems.

This notebook demonstrates, with a minimal worked example:

1. How Data Envelopment Analysis (DEA) constructs an efficient frontier from a set of comparable Decision-Making Units (DMUs) — three fictional flow stations, two inputs, one output.
2. How input-oriented CCR efficiency scores are computed by solving one linear program per DMU.
3. Why the method fails for isolated facilities — shown by directly violating each of DEA's three requirements: **peers**, **comparability**, and **variation**.

All data are fictional but field-plausible. Requirements: `numpy`, `pandas`, `matplotlib`, `scipy` (≥1.6 for the HiGHS LP solver).

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog
import matplotlib.pyplot as plt

pd.set_option("display.precision", 4)

## 2. Three fictional DMUs

Each DMU is an onshore flow station described by two inputs and one output:

| Variable | Role | Unit |
|---|---|---|
| `gas_lift_mscfd` | Input 1 — gas-lift injection volume | Mscf/d |
| `operating_hours` | Input 2 — monthly operating hours | h |
| `oil_bopd` | Output — oil production | bopd |

The stations operate under broadly similar conditions, so the comparison is meaningful — this is the assumption Section 5 will deliberately break.

In [ ]:
data = pd.DataFrame(
    {
        "DMU": ["FS-Alpha", "FS-Bravo", "FS-Charlie"],
        "gas_lift_mscfd": [1200.0, 900.0, 1500.0],
        "operating_hours": [720.0, 700.0, 740.0],
        "oil_bopd": [1500.0, 1400.0, 1350.0],
    }
).set_index("DMU")

data

## 3. The efficiency question, stated per unit of output

With multiple inputs, no single ratio settles the question. FS-Bravo uses the least gas lift; FS-Alpha produces the most oil. Normalising inputs per barrel of daily production makes the trade-off visible: each DMU becomes a point in *input-per-output* space, and better performance means being closer to the origin.

In [ ]:
X = data[["gas_lift_mscfd", "operating_hours"]].values  # inputs, shape (n, m)
Y = data[["oil_bopd"]].values                            # output, shape (n, 1)

per_output = pd.DataFrame(
    {
        "gas_lift_per_bopd": X[:, 0] / Y[:, 0],
        "hours_per_bopd": X[:, 1] / Y[:, 0],
    },
    index=data.index,
)
per_output

## 4. Efficiency scores via linear programming (input-oriented CCR)

For each DMU *o*, the input-oriented CCR model asks: *by what uniform factor θ could DMU o contract its inputs and still be dominated by some non-negative combination of the observed DMUs?*

$$
\min_{\theta, \lambda} \; \theta
\quad \text{s.t.} \quad
\sum_j \lambda_j x_{ij} \le \theta \, x_{io} \;\; \forall i,
\qquad
\sum_j \lambda_j y_{rj} \ge y_{ro} \;\; \forall r,
\qquad
\lambda_j \ge 0 .
$$

One LP is solved per DMU — each unit gets the most favourable feasible comparison. θ = 1 places the unit on the frontier; θ < 1 measures the feasible proportional input reduction.

In [ ]:
def dea_ccr_input(X, Y):
    """Input-oriented CCR DEA.

    Parameters
    ----------
    X : array-like, shape (n, m) — input matrix
    Y : array-like, shape (n, s) — output matrix

    Returns
    -------
    scores : ndarray, shape (n,) — efficiency score theta for each DMU
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    n, m = X.shape
    s = Y.shape[1]
    scores = np.zeros(n)

    for o in range(n):
        # Decision variables: [theta, lambda_1, ..., lambda_n]
        c = np.zeros(n + 1)
        c[0] = 1.0  # minimise theta

        A_ub, b_ub = [], []

        # Input constraints:  sum_j lambda_j * x_ij - theta * x_io <= 0
        for i in range(m):
            row = np.zeros(n + 1)
            row[0] = -X[o, i]
            row[1:] = X[:, i]
            A_ub.append(row)
            b_ub.append(0.0)

        # Output constraints: -sum_j lambda_j * y_rj <= -y_ro
        for r in range(s):
            row = np.zeros(n + 1)
            row[1:] = -Y[:, r]
            A_ub.append(row)
            b_ub.append(-Y[o, r])

        bounds = [(None, None)] + [(0, None)] * n
        res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                      bounds=bounds, method="highs")
        if not res.success:
            raise RuntimeError(f"LP failed for DMU {o}: {res.message}")
        scores[o] = res.fun

    return scores

In [ ]:
scores = dea_ccr_input(X, Y)

results = data.copy()
results["efficiency"] = scores.round(4)
results

FS-Alpha and FS-Bravo sit on the frontier (θ = 1). FS-Charlie scores ≈ 0.876: the frontier evidence implies it could produce its current output with roughly 87.6% of each input. The score also quantifies what closing the gap would look like:

In [ ]:
results["target_gas_lift_mscfd"] = (X[:, 0] * scores).round(1)
results["target_operating_hours"] = (X[:, 1] * scores).round(1)
results[["efficiency", "target_gas_lift_mscfd", "target_operating_hours"]]

### The efficient frontier, visually

In input-per-output space the efficient units define the boundary of demonstrated performance; inefficient units lie above/right of it. Distance to the frontier is the efficiency gap.

In [ ]:
x1 = X[:, 0] / Y[:, 0]   # gas lift per bopd
x2 = X[:, 1] / Y[:, 0]   # hours per bopd
eff_mask = scores > 1 - 1e-6

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.scatter(x1[eff_mask], x2[eff_mask], s=90, zorder=3, label="Efficient (θ = 1)")
ax.scatter(x1[~eff_mask], x2[~eff_mask], s=90, zorder=3, marker="s",
           label="Inefficient (θ < 1)")

frontier_pts = sorted(zip(x1[eff_mask], x2[eff_mask]))
ax.plot([p[0] for p in frontier_pts], [p[1] for p in frontier_pts],
        lw=2, zorder=2, label="Efficient frontier")

for name, xi, yi, th in zip(data.index, x1, x2, scores):
    ax.annotate(f"{name}  (θ = {th:.3f})", (xi, yi),
                textcoords="offset points", xytext=(10, 6))

ax.set_xlabel("Gas lift per unit output  (Mscf/d per bopd)")
ax.set_ylabel("Operating hours per unit output  (h per bopd)")
ax.set_title("Input-oriented view: closer to the origin is better")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Why isolated facilities break it

DEA has three non-negotiable requirements: **peers**, **comparability**, and **variation**. Isolated facilities violate all three — structurally, not accidentally. Each violation is reproduced below.

### 5.1 No peers — a single DMU

With one facility there is one data point. The LP still solves, but the unit is trivially compared against itself:

In [ ]:
single_score = dea_ccr_input(X[:1], Y[:1])
print(f"Efficiency score with n = 1:  {single_score[0]:.4f}")

θ = 1.0 — a "perfect" score carrying zero information. A frontier built from one observation is not a frontier; it is a point. The score does not certify efficiency; it records the absence of any basis for comparison.

### 5.2 No comparability — a structural outlier in the set

Suppose a deep-water platform (`DW-Delta`) is forced into the comparison set. It is not badly run and not better run — it is structurally different: far higher throughput per unit input because of reservoir and configuration, not management.

In [ ]:
outlier = pd.DataFrame(
    {"gas_lift_mscfd": [4000.0], "operating_hours": [720.0], "oil_bopd": [9000.0]},
    index=pd.Index(["DW-Delta"], name="DMU"),
)
data_mixed = pd.concat([data, outlier])

X2 = data_mixed[["gas_lift_mscfd", "operating_hours"]].values
Y2 = data_mixed[["oil_bopd"]].values
scores_mixed = dea_ccr_input(X2, Y2)

comparison = pd.DataFrame(
    {
        "score_before": list(scores.round(3)) + [np.nan],
        "score_with_outlier": scores_mixed.round(3),
    },
    index=data_mixed.index,
)
comparison

The frontier is now defined almost entirely by the structural outlier. Every onshore station collapses to a low score — FS-Alpha and FS-Bravo, genuinely efficient within their own class, drop to ≈ 0.56 and ≈ 0.69. The scores no longer measure operational performance; they measure configuration difference. This is how good operations end up looking bad.

### 5.3 No variation — identical DMUs

Finally, suppose three units perform identically:

In [ ]:
X3 = np.tile([1000.0, 720.0], (3, 1))
Y3 = np.tile([1400.0], (3, 1))
scores_identical = dea_ccr_input(X3, Y3)
print("Scores with zero variation:", scores_identical.round(4))

All units score 1.0. The frontier exists but discriminates nothing — there is no observed "better" to learn from. DEA identifies good performance from spread; remove the spread and the answer is uniform and uninformative.

### Summary of the three failures

| Requirement | Violation | Result |
|---|---|---|
| Peers | n = 1 | θ = 1 trivially; no frontier, no information |
| Comparability | Structural outlier in set | Frontier pulled to the outlier; scores become configuration artefacts |
| Variation | Identical units | All θ = 1; frontier has no discriminating power |

The method does not degrade gracefully in these conditions — it produces numbers that look like answers while measuring nothing. The environment does not make DEA difficult to apply; it removes the conditions the method requires to produce meaningful results.

## 6. The construction question

A valid comparison set would need to satisfy the same three requirements demonstrated above: **comparable** (distance from the frontier reflects efficiency, not structure), **varied** (enough spread to draw a frontier with discriminating power), and **bounded** (representing what the facility could actually achieve, not a theoretical ideal).

If real peers don't exist — can the peers be constructed?

That question is the subject of this series and of research currently under peer review (preprint on Research Square).

- Article: [The Method That Almost Works](https://www.linkedin.com/pulse/method-almost-works-oghenero-inana-za0oe)
- Repository: `pasy-dea-analytics` — further notebooks extend this example toward synthetic peer construction.